# Calculate Scale and Zero Point

### Import Required Libraries

In [1]:
import torch
import numpy as np

### Calculate Scale & Zero Point

In [2]:
def calculate_scale_zero_point(tensor, qmin=-128, qmax=127):
    rmin, rmax = float(tensor.min().item()), float(tensor.max().item())
    if rmin == rmax:
        scale =  1.0
    else:
        scale = (rmax - rmin) / (qmax - qmin)
    zero_point = int(round(qmin - (rmin / scale)))
    if zero_point < qmin:
        zero_point = qmin
    elif zero_point > qmax:
        zero_point = qmax
    return scale, zero_point

### Quantization

In [3]:
def quantize_tensor(tensor, scale, zero_point):
    scaled_and_shifted_tensor = tensor / scale + zero_point
    rounded_tensor = torch.round(scaled_and_shifted_tensor)
    q_min = torch.iinfo(torch.int8).min
    q_max = torch.iinfo(torch.int8).max
    q_tensor = rounded_tensor.clamp(q_min,q_max).to(torch.int8)
    return q_tensor

### Dequantization

In [4]:
def dequantize_tensor(quantized_tensor, scale, zero_point):
    dequantized_tensor = scale * (quantized_tensor.float() - zero_point)
    return dequantized_tensor

### Tensor 1 : Normal Mixed Range

In [7]:
t1 = np.array([-1.5, -0.8, 0.0, 0.9, 2.3], dtype=np.float32)
t1 = torch.tensor(t1)
scale, zero_point = calculate_scale_zero_point(t1)
quantized = quantize_tensor(t1, scale, zero_point)
dequantized = dequantize_tensor(quantized, scale, zero_point)
abs_error = torch.abs(t1 - dequantized)
mean_abs_error = torch.mean(abs_error).item()

In [8]:
print("---------Tensor 1: Normal Mixed Range---------")
print("Tensor values: ",t1)
print("Tensor min: ",t1.min().item())
print("Tensor max: ",t1.max().item())
print("Scale: ",scale)
print("Zero point: ",zero_point)
print("Quantized tensor: ",quantized)
print("Dequantized tensor: ",dequantized)
print("Mean Absolute Error: ",mean_abs_error)

---------Tensor 1: Normal Mixed Range---------
Tensor values:  tensor([-1.5000, -0.8000,  0.0000,  0.9000,  2.3000])
Tensor min:  -1.5
Tensor max:  2.299999952316284
Scale:  0.014901960597318761
Zero point:  -27
Quantized tensor:  tensor([-128,  -81,  -27,   33,  127], dtype=torch.int8)
Dequantized tensor:  tensor([-1.5051, -0.8047,  0.0000,  0.8941,  2.2949])
Mean Absolute Error:  0.004156863782554865


### Tensor 2 : All positive values

In [9]:
t2 = np.array([0.1, 0.5, 1.2, 2.0, 3.5], dtype=np.float32)
t2 = torch.tensor(t2)
scale, zero_point = calculate_scale_zero_point(t2)
quantized = quantize_tensor(t2, scale, zero_point)
dequantized = dequantize_tensor(quantized, scale, zero_point)
abs_error = torch.abs(t2 - dequantized)
mean_abs_error = torch.mean(abs_error)

In [10]:
print("---------Tensor 2: All positive values---------")
print("Tensor values: ",t2)
print("Tensor min: ",t2.min().item())
print("Tensor max: ",t2.max().item())
print("Scale: ",scale)
print("Zero point: ",zero_point)
print("Quantized tensor: ",quantized)
print("Dequantized tensor: ",dequantized)
print("Mean Absolute Error: ",mean_abs_error)

---------Tensor 2: All positive values---------
Tensor values:  tensor([0.1000, 0.5000, 1.2000, 2.0000, 3.5000])
Tensor min:  0.10000000149011612
Tensor max:  3.5
Scale:  0.013333333327489741
Zero point:  -128
Quantized tensor:  tensor([-120,  -90,  -38,   22,  127], dtype=torch.int8)
Dequantized tensor:  tensor([0.1067, 0.5067, 1.2000, 2.0000, 3.4000])
Mean Absolute Error:  tensor(0.0227)


### Tensor 3: All negative values

In [11]:
t3 = np.array([-3.0, -2.1, -1.4, -0.6, -0.1], dtype=np.float32) 
t3 = torch.tensor(t3)
scale, zero_point = calculate_scale_zero_point(t3)
quantized = quantize_tensor(t3, scale, zero_point)
dequantized = dequantize_tensor(quantized, scale, zero_point)
abs_error = torch.abs(t3 - dequantized)
mean_abs_error = torch.mean(abs_error)

In [12]:
print("---------Tensor 3: All negative values---------")
print("Tensor values: ",t3)
print("Tensor min: ",t3.min().item())
print("Tensor max: ",t3.max().item())
print("Scale: ",scale)
print("Zero point: ",zero_point)
print("Quantized tensor: ",quantized)
print("Dequantized tensor: ",dequantized)
print("Mean Absolute Error: ",mean_abs_error)

---------Tensor 3: All negative values---------
Tensor values:  tensor([-3.0000, -2.1000, -1.4000, -0.6000, -0.1000])
Tensor min:  -3.0
Tensor max:  -0.10000000149011612
Scale:  0.011372549013764251
Zero point:  127
Quantized tensor:  tensor([-128,  -58,    4,   74,  118], dtype=torch.int8)
Dequantized tensor:  tensor([-2.9000, -2.1039, -1.3988, -0.6027, -0.1024])
Mean Absolute Error:  tensor(0.0220)


### Tensor 4: Single unique value (constant tensor)

In [13]:
t4 = np.array([5.0, 5.0, 5.0], dtype=np.float32) 
t4 = torch.tensor(t4)
scale, zero_point = calculate_scale_zero_point(t4)
quantized = quantize_tensor(t4, scale, zero_point)
dequantized = dequantize_tensor(quantized, scale, zero_point)
abs_error = torch.abs(t4 - dequantized)
mean_abs_error = torch.mean(abs_error)

In [14]:
print("---------Tensor 4: Single unique value (constant tensor)---------")
print("Tensor values: ",t4)
print("Tensor min: ",t4.min().item())
print("Tensor max: ",t4.max().item())
print("Scale: ",scale)
print("Zero point: ",zero_point)
print("Quantized tensor: ",quantized)
print("Dequantized tensor: ",dequantized)
print("Mean Absolute Error: ",mean_abs_error)

---------Tensor 4: Single unique value (constant tensor)---------
Tensor values:  tensor([5., 5., 5.])
Tensor min:  5.0
Tensor max:  5.0
Scale:  1.0
Zero point:  -128
Quantized tensor:  tensor([-123, -123, -123], dtype=torch.int8)
Dequantized tensor:  tensor([5., 5., 5.])
Mean Absolute Error:  tensor(0.)


### Tensor 5: Very small floating-point values

In [15]:
t5 = np.array([1e-9, 2e-9, -1e-9], dtype=np.float32) 
t5 = torch.tensor(t5)
scale, zero_point = calculate_scale_zero_point(t5)
quantized = quantize_tensor(t5, scale, zero_point)
dequantized = dequantize_tensor(quantized, scale, zero_point)
abs_error = torch.abs(t5 - dequantized)
mean_abs_error = torch.mean(abs_error)

In [16]:
print("---------Tensor 5: Very small floating-point values---------")
print("Tensor values: ",t5)
print("Tensor min: ",t5.min().item())
print("Tensor max: ",t5.max().item())
print("Scale: ",scale)
print("Zero point: ",zero_point)
print("Quantized tensor: ",quantized)
print("Dequantized tensor: ",dequantized)
print("Mean Absolute Error: ",mean_abs_error)

---------Tensor 5: Very small floating-point values---------
Tensor values:  tensor([ 1.0000e-09,  2.0000e-09, -1.0000e-09])
Tensor min:  -9.999999717180685e-10
Tensor max:  1.999999943436137e-09
Scale:  1.1764705549624336e-11
Zero point:  -43
Quantized tensor:  tensor([  42,  127, -128], dtype=torch.int8)
Dequantized tensor:  tensor([ 1.0000e-09,  2.0000e-09, -1.0000e-09])
Mean Absolute Error:  tensor(0.)
